# 06 ADR騰落率 → 日経平均予測
# Nikkei 225 Prediction from ADR Returns

## 概要 / Overview

日本時間 8 時（米国市場クローズ後）に入手できる情報を使い、その日の日経平均を予測します。  
We predict Nikkei 225 using information available by 8 AM JST (after US market close).

## 予測ターゲット / Prediction Targets

| ターゲット | 計算式 | 意味 |
|------------|--------|----- |
| **始値リターン** | `open_t / close_{t-1} - 1` | 翌朝の窓開き |
| **終値リターン** | `close_t / close_{t-1} - 1` | 丸一日のリターン |
| **場中リターン** | `close_t / open_t - 1` | 開場後の動き |

場中リターンを追加することで「**一夜の情報は始値にどこまで織り込まれるか**」を検証できます。

## 予測手法 / Approaches

| # | 手法 | 概要 |
|---|------|------|
| 1 | **ルールベース合成日経** | ADRリターン × 価格ウェイト → 日経リターン推定 |
| 2 | **回帰モデル** | ADR + USDJPY + 米国指数 → Ridge / XGBoost |

## 入力データ（8時までに入手可能）

| データ | ソース |
|--------|--------|
| 日経上位構成銘柄の ADR 終値 | yfinance OTC/NYSE |
| USD/JPY レート | yfinance `JPY=X` |
| S&P500 / NASDAQ / VIX | yfinance |

## セクション構成

| # | 内容 |
|---|------|
| 0 | 環境セットアップ |
| 1 | データ取得 |
| 2 | 日付アライメントと特徴量エンジニアリング |
| 3 | Idea 1: ルールベース合成日経 |
| 4 | Idea 2: 回帰モデル |
| 5 | 両手法の比較と考察 |


In [ ]:
# -- Step 0: Install packages -------------------------------------------
import subprocess, sys

pkgs = ['yfinance', 'scikit-learn>=1.5.0', 'xgboost>=2.0.0']
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q'] + pkgs,
    capture_output=True, text=True
)
out = result.stdout + result.stderr
print(out.strip() if out.strip() else 'All packages already up to date.')


In [ ]:
# -- Step 0b: Repository & path setup ------------------------------------
import os, sys

REPO = '/content/Nikkei_Analysis'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists(REPO):
        !git clone https://github.com/Takumi-Itokawa-Finance/Nikkei_Analysis.git {REPO}
    else:
        !git -C {REPO} pull
    os.chdir(REPO)

    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DATA = '/content/drive/MyDrive/Nikkei_Analysis/data'
    os.makedirs(DRIVE_DATA, exist_ok=True)
    if not os.path.exists(f'{REPO}/data'):
        os.symlink(DRIVE_DATA, f'{REPO}/data')

    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    print(f'Setup complete.  CWD={os.getcwd()}')
else:
    print('Running in local environment.')


## 1. データ取得 / Data Collection

### タイムゾーンの関係

```
月曜日 (US)  ──────────────────────────────▶
         US市場クローズ 4PM ET = 火曜 6AM JST
                              ↓ ← 8時時点で入手可能
火曜日 (日本) ──▶ 東証オープン 9AM JST ──▶ クローズ 3:30PM JST
```

**US の日付 T のデータが、日本の日付 T+1 の予測特徴量になります。**

### 対象銘柄

日経225の上位構成銘柄（プライスウェイト上位）と対応する ADR：

| 日本株 | 会社 | ADR | 市場 |
|--------|------|-----|------|
| 9983.T | ファーストリテイリング | FRCOY | OTC |
| 8035.T | 東京エレクトロン | TOELY | OTC |
| 9984.T | ソフトバンクG | SFTBY | OTC |
| 6954.T | ファナック | FANUY | OTC |
| 9433.T | KDDI | KDDIY | OTC |
| 4063.T | 信越化学 | SHECY | OTC |
| 6861.T | キーエンス | KYCCF | OTC |
| 6367.T | ダイキン | DKILY | OTC |
| 6857.T | アドバンテスト | ATEYY | OTC |
| 6758.T | ソニーG | SONY | NYSE |


In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import yfinance as yf

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})

OUTPUT_FIGS = 'output/figures'
os.makedirs(OUTPUT_FIGS, exist_ok=True)

PERIOD = '3y'

# Japanese stock tickers and corresponding ADR tickers
JP_ADR_MAP = {
    '9983.T': 'FRCOY',
    '8035.T': 'TOELY',
    '9984.T': 'SFTBY',
    '6954.T': 'FANUY',
    '9433.T': 'KDDIY',
    '4063.T': 'SHECY',
    '6861.T': 'KYCCF',
    '6367.T': 'DKILY',
    '6857.T': 'ATEYY',
    '6758.T': 'SONY',
}

JP_NAMES = {
    '9983.T': 'FastRetailing',
    '8035.T': 'TokyoElectron',
    '9984.T': 'SoftBank',
    '6954.T': 'Fanuc',
    '9433.T': 'KDDI',
    '4063.T': 'ShinEtsu',
    '6861.T': 'Keyence',
    '6367.T': 'Daikin',
    '6857.T': 'Advantest',
    '6758.T': 'Sony',
}

JP_TICKERS  = list(JP_ADR_MAP.keys())
ADR_TICKERS = list(JP_ADR_MAP.values())
NAMES       = list(JP_NAMES.values())

US_MACRO = {'SP500': '^GSPC', 'NASDAQ': '^IXIC', 'VIX': '^VIX', 'USDJPY': 'JPY=X'}


In [ ]:
def _clean(df):
    # Flatten MultiIndex columns (yfinance >= 0.2.x returns Price x Ticker levels)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    # Strip timezone so all indices are tz-naive
    idx = pd.to_datetime(df.index)
    df.index = idx.tz_localize(None) if idx.tz is not None else idx
    return df

def fetch_ohlc(tickers, period, label):
    frames = {}
    failed = []
    for t in tickers:
        df = yf.download(t, period=period, interval='1d', progress=False, auto_adjust=True)
        if len(df) > 10:
            frames[t] = _clean(df)[['Open', 'Close']].copy()
        else:
            failed.append(t)
    if failed:
        print(f'[WARN] {label}: no data for {failed}')
    return frames

def fetch_close(tickers_dict, period):
    frames = {}
    for name, ticker in tickers_dict.items():
        df = yf.download(ticker, period=period, interval='1d', progress=False, auto_adjust=True)
        if len(df) > 10:
            frames[name] = _clean(df)['Close'].squeeze()
        else:
            print(f'[WARN] No data for {name} ({ticker})')
    return pd.DataFrame(frames)

print('Fetching Nikkei 225 OHLC ...')
nikkei_raw = _clean(yf.download('^N225', period=PERIOD, interval='1d', progress=False, auto_adjust=True))

print('Fetching Japanese stocks (OHLC for price-weight) ...')
jp_ohlc = fetch_ohlc(JP_TICKERS, PERIOD, 'JP stocks')

print('Fetching ADRs ...')
adr_ohlc = fetch_ohlc(ADR_TICKERS, PERIOD, 'ADRs')

print('Fetching US macro ...')
us_df = fetch_close(US_MACRO, PERIOD)

print('Done.')
print(f'Nikkei rows : {len(nikkei_raw)}')
print(f'JP stocks   : {len(jp_ohlc)} / {len(JP_TICKERS)}')
print(f'ADRs        : {len(adr_ohlc)} / {len(ADR_TICKERS)}')


In [ ]:
# Check ADR data coverage
rows = []
for jp_t, adr_t in JP_ADR_MAP.items():
    name   = JP_NAMES[jp_t]
    jp_ok  = jp_t  in jp_ohlc
    adr_ok = adr_t in adr_ohlc
    n_adr  = len(adr_ohlc[adr_t]) if adr_ok else 0
    rows.append({'Name': name, 'JP ticker': jp_t, 'ADR ticker': adr_t,
                 'JP OK': jp_ok, 'ADR OK': adr_ok, 'ADR rows': n_adr})

cov_df = pd.DataFrame(rows)
display(cov_df)

# Keep only pairs where both JP and ADR data exist
valid_pairs = [
    (jp_t, adr_t, JP_NAMES[jp_t])
    for jp_t, adr_t in JP_ADR_MAP.items()
    if jp_t in jp_ohlc and adr_t in adr_ohlc
]
print(f'\nValid pairs for analysis: {len(valid_pairs)} / {len(JP_ADR_MAP)}')
for jp_t, adr_t, name in valid_pairs:
    print(f'  {name:20s}  {jp_t} -> {adr_t}')


## 2. 日付アライメントと特徴量エンジニアリング

### 日付シフトの方針

| データ | 日付ラベル | 処理 |
|--------|------------|------|
| ADR・US指数 | US日付 T | **+1営業日シフト** → 日本日付 T+1 の特徴量として使用 |
| USDJPY | US日付 T | 同上 |
| 日経225 | 日本日付 T | ターゲット変数（シフトなし） |

### FX調整

ADR は USD 建てのため、日本株の JPY リターンに換算します：

$$\text{Japan stock return (JPY)} \approx \text{ADR return (USD)} + \text{USDJPY return}$$

`JPY=X`（yfinance）は JPY/USD 表示（上昇 = 円安）。


In [ ]:
# -- Nikkei targets -------------------------------------------------------
nk = nikkei_raw[['Open', 'Close']].copy().dropna()
nk.index = pd.to_datetime(nk.index)

nk['open_ret']     = nk['Open']  / nk['Close'].shift(1) - 1  # overnight gap
nk['close_ret']    = nk['Close'] / nk['Close'].shift(1) - 1  # full-day return
nk['intraday_ret'] = nk['Close'] / nk['Open']  - 1            # intraday move
nk = nk.dropna()

print('Nikkei targets:')
print(nk[['open_ret', 'close_ret', 'intraday_ret']].describe().T
      [['mean', 'std', 'min', 'max']]
      .rename(columns={'mean':'Mean','std':'Std','min':'Min','max':'Max'}))


In [ ]:
# -- ADR returns (USD) + USDJPY return ------------------------------------
usdjpy_close = us_df['USDJPY'].dropna()
usdjpy_ret   = usdjpy_close.pct_change()  # positive = yen weakens

adr_ret_usd = {}
for jp_t, adr_t, name in valid_pairs:
    close_s = adr_ohlc[adr_t]['Close'].squeeze().dropna()
    adr_ret_usd[name] = close_s.pct_change()

adr_ret_df = pd.DataFrame(adr_ret_usd)
adr_ret_df.index = pd.to_datetime(adr_ret_df.index)

# FX-adjust: JPY return = ADR_return(USD) + USDJPY_return
adr_ret_jpy = adr_ret_df.add(usdjpy_ret, axis=0)

print(f'ADR return (USD) shape : {adr_ret_df.shape}')
print(f'ADR return (JPY) shape : {adr_ret_jpy.shape}')
print()
print('Mean ADR-JPY return vs mean actual JP stock return:')
for jp_t, adr_t, name in valid_pairs:
    jp_ret  = jp_ohlc[jp_t]['Close'].squeeze().pct_change().mean() * 100
    adr_ret = adr_ret_jpy[name].mean() * 100
    print(f'  {name:20s}  JP={jp_ret:+.3f}%  ADR(JPY)={adr_ret:+.3f}%')


In [ ]:
# -- US macro returns ------------------------------------------------------
us_macro_ret = us_df[['SP500', 'NASDAQ', 'VIX']].pct_change()
us_macro_ret['USDJPY'] = usdjpy_ret

# -- Shift US data +1 business day to align with Japan dates ---------------
def shift_us_to_jp(df):
    """Forward-shift US-dated data by 1 business day (US date T -> Japan date T+1)."""
    shifted = df.copy()
    shifted.index = df.index + pd.tseries.offsets.BDay(1)
    return shifted

adr_jpy_shifted  = shift_us_to_jp(adr_ret_jpy)
us_macro_shifted = shift_us_to_jp(us_macro_ret)

# -- Merge on Japan dates --------------------------------------------------
targets = nk[['open_ret', 'close_ret', 'intraday_ret']]

feat = targets.join(adr_jpy_shifted, how='inner')
feat = feat.join(us_macro_shifted,   how='inner')
feat = feat.dropna()

TARGET_COLS  = ['open_ret', 'close_ret', 'intraday_ret']
ADR_COLS     = [name for _, _, name in valid_pairs]
MACRO_COLS   = ['SP500', 'NASDAQ', 'VIX', 'USDJPY']
FEATURE_COLS = ADR_COLS + MACRO_COLS

print(f'Merged dataset : {feat.shape}')
print(f'Period         : {feat.index[0].date()} to {feat.index[-1].date()}')
print(f'Feature cols   : {FEATURE_COLS}')


In [ ]:
# -- Correlation heatmap: features vs targets ------------------------------
corr = feat[FEATURE_COLS + TARGET_COLS].corr()[TARGET_COLS].loc[FEATURE_COLS]

fig, ax = plt.subplots(figsize=(7, len(FEATURE_COLS) * 0.55 + 1))
im = ax.imshow(corr.values, cmap='RdBu', vmin=-0.8, vmax=0.8, aspect='auto')
ax.set_xticks(range(len(TARGET_COLS)))
ax.set_xticklabels(['始値\nopen_ret', '終値\nclose_ret', '場中\nintraday_ret'], fontsize=10)
ax.set_yticks(range(len(FEATURE_COLS)))
ax.set_yticklabels(FEATURE_COLS, fontsize=9)
for i in range(len(FEATURE_COLS)):
    for j in range(len(TARGET_COLS)):
        ax.text(j, i, f'{corr.values[i, j]:.2f}',
                ha='center', va='center', fontsize=8,
                color='white' if abs(corr.values[i, j]) > 0.5 else 'black')
plt.colorbar(im, ax=ax, label='Pearson r')
ax.set_title('Feature-Target Correlation  (ADR/Macro → Nikkei Returns)')
plt.tight_layout()
plt.savefig(f'{OUTPUT_FIGS}/06_01_correlation.png', dpi=150)
plt.show()

print('\n始値リターンとの相関（降順）:')
print(corr['open_ret'].abs().sort_values(ascending=False).to_string())


## 3. Idea 1: ルールベース合成日経

### 方針

日経225はプライスウェイト方式（ダウ平均と同じ）なので、  
各銘柄のリターンを **前日の株価ウェイト** で加重平均すれば日経リターンを近似できます。

$$\hat{r}_{\text{Nikkei}} = \sum_i w_i \cdot r_i^{\text{JPY}}, \quad w_i = \frac{P_i^{\text{prev}}}{\sum_j P_j^{\text{prev}}}$$

- $P_i^{\text{prev}}$: 前日の日本株終値
- $r_i^{\text{JPY}}$: ADR リターン（JPY 換算）

> **注意**: 上位 10 銘柄のみカバー（日経全体の約 60-70%）のため、  
> 残り 30-40% は推定できず、結果には系統的な誤差が含まれます。


In [ ]:
# Build JP stock price matrix (for weight calculation)
jp_close_dict = {}
for jp_t, adr_t, name in valid_pairs:
    s = jp_ohlc[jp_t]['Close'].squeeze().dropna()
    s.index = pd.to_datetime(s.index)
    jp_close_dict[name] = s

jp_close = pd.DataFrame(jp_close_dict)

# Previous-day JP close price on each Japan date (already JP-dated, just shift by 1 row)
jp_price_prev = jp_close.shift(1)

# Normalised price weights (sum to 1 across available stocks)
total_prev = jp_price_prev.sum(axis=1)
jp_weights = jp_price_prev.div(total_prev, axis=0)  # each row sums to 1

# Align with feature dataset dates
jp_weights_aligned = jp_weights.reindex(feat.index).fillna(method='ffill')

# Synthetic Nikkei return = weighted sum of ADR-JPY returns
adr_jpy_aligned = feat[ADR_COLS]
common_names = [n for n in ADR_COLS if n in jp_weights_aligned.columns]

w = jp_weights_aligned[common_names]
r = adr_jpy_aligned[common_names]
synthetic_ret = (w * r).sum(axis=1)  # price-weighted synthetic return

print(f'Common stocks for synthesis: {common_names}')
print(f'Mean weight coverage check (should sum ~1): {w.sum(axis=1).mean():.3f}')
print()
print(f'Synthetic return stats:')
print(synthetic_ret.describe())


In [ ]:
def eval_metrics(y_true, y_pred, label):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    y_true, y_pred = y_true[mask], y_pred[mask]
    rmse    = np.sqrt(np.mean((y_pred - y_true) ** 2))
    corr    = np.corrcoef(y_pred, y_true)[0, 1]
    dir_acc = (np.sign(y_pred) == np.sign(y_true)).mean()
    return {'Target': label, 'RMSE': rmse, 'Correlation': corr, 'Dir. Accuracy': dir_acc}

idea1_rows = [
    eval_metrics(feat['open_ret'],     synthetic_ret, '始値 open_ret'),
    eval_metrics(feat['close_ret'],    synthetic_ret, '終値 close_ret'),
    eval_metrics(feat['intraday_ret'], synthetic_ret, '場中 intraday_ret'),
]
idea1_df = pd.DataFrame(idea1_rows).set_index('Target')

print('=== Idea 1: Rule-based Synthetic Nikkei ===')
display(
    idea1_df.style
    .format({'RMSE': '{:.4f}', 'Correlation': '{:.3f}', 'Dir. Accuracy': '{:.3f}'})
    .background_gradient(subset=['Dir. Accuracy'], cmap='RdYlGn')
    .background_gradient(subset=['Correlation'],   cmap='RdYlGn')
)


In [ ]:
# -- Scatter: synthetic vs actual for open and close ----------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col, title, color in zip(
    axes,
    ['open_ret', 'close_ret', 'intraday_ret'],
    ['始値リターン (open_ret)', '終値リターン (close_ret)', '場中リターン (intraday_ret)'],
    ['steelblue', 'orange', 'grey'],
):
    x = synthetic_ret
    y = feat[col]
    mask = x.notna() & y.notna()
    ax.scatter(x[mask], y[mask], alpha=0.35, s=15, color=color)

    # OLS line
    m, b = np.polyfit(x[mask], y[mask], 1)
    xl = np.linspace(x[mask].min(), x[mask].max(), 100)
    ax.plot(xl, m * xl + b, color='black', linewidth=1.5)

    r = np.corrcoef(x[mask], y[mask])[0, 1]
    da = (np.sign(x[mask]) == np.sign(y[mask])).mean()
    ax.set_title(f'{title}\nr={r:.3f}  Dir.Acc={da:.3f}', fontsize=10)
    ax.set_xlabel('Synthetic Nikkei return (rule-based)')
    ax.set_ylabel('Actual Nikkei return')
    ax.axhline(0, color='grey', linewidth=0.5, linestyle='--')
    ax.axvline(0, color='grey', linewidth=0.5, linestyle='--')

plt.suptitle('Idea 1: Rule-based Synthetic Nikkei vs Actual', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_FIGS}/06_02_idea1_scatter.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# -- Time series: synthetic vs actual open_ret (rolling 60d window) -------
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

plot_idx = feat.index

ax = axes[0]
ax.plot(plot_idx, feat['open_ret'] * 100,   color='navy',  linewidth=0.8, label='Actual open_ret')
ax.plot(plot_idx, synthetic_ret * 100, color='orange', linewidth=0.8,
        linestyle='--', label='Synthetic (rule-based)')
ax.set_ylabel('Return (%)')
ax.set_title('Idea 1: 始値リターン — Actual vs Synthetic')
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

ax = axes[1]
ax.plot(plot_idx, feat['close_ret'] * 100,  color='navy',  linewidth=0.8, label='Actual close_ret')
ax.plot(plot_idx, synthetic_ret * 100, color='orange', linewidth=0.8,
        linestyle='--', label='Synthetic (rule-based)')
ax.set_ylabel('Return (%)')
ax.set_title('Idea 1: 終値リターン — Actual vs Synthetic')
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.tight_layout()
plt.savefig(f'{OUTPUT_FIGS}/06_03_idea1_timeseries.png', dpi=150)
plt.show()


## 4. Idea 2: 回帰モデル

ADR リターン（JPY換算）＋ 米国マクロ変数を特徴量として、  
始値・終値・場中リターンをそれぞれ予測します。

| モデル | 特徴 |
|--------|------|
| Ridge | 線形・正則化あり。特徴量の係数で寄与を解釈 |
| XGBoost | 非線形・特徴間の相互作用を捉える |

**評価**: ウォークフォワード交差検証（時系列スプリット、シャッフルなし）


In [ ]:
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor

X = feat[FEATURE_COLS].values
TARGET_LABELS = {
    'open_ret':     '始値リターン',
    'close_ret':    '終値リターン',
    'intraday_ret': '場中リターン',
}

ALPHAS   = [0.01, 0.1, 1, 10, 100, 500]
TSCV     = TimeSeriesSplit(n_splits=5)
TEST_SIZE = 252  # last 1 year as hold-out

X_train, X_test = X[:-TEST_SIZE], X[-TEST_SIZE:]

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_train)
X_te_s = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]} days  |  Test: {X_test.shape[0]} days')
print(f'Features: {FEATURE_COLS}')


In [ ]:
idea2_rows = []
models_dict = {}  # store fitted models for feature importance

for target_col, target_label in TARGET_LABELS.items():
    y = feat[target_col].values
    y_train, y_test = y[:-TEST_SIZE], y[-TEST_SIZE:]

    # -- Ridge -----------------------------------------------------------
    ridge = RidgeCV(alphas=ALPHAS, cv=TSCV)
    ridge.fit(X_tr_s, y_train)
    ridge_pred = ridge.predict(X_te_s)

    # -- XGBoost ---------------------------------------------------------
    val_n = int(len(X_train) * 0.2)
    xgb = XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=3,
        subsample=0.8, colsample_bytree=0.8,
        early_stopping_rounds=20, eval_metric='rmse',
        random_state=42, verbosity=0,
    )
    xgb.fit(
        X_tr_s[:-val_n], y_train[:-val_n],
        eval_set=[(X_tr_s[-val_n:], y_train[-val_n:])],
        verbose=False,
    )
    xgb_pred = xgb.predict(X_te_s)

    models_dict[target_col] = {'ridge': ridge, 'xgb': xgb,
                               'y_test': y_test,
                               'ridge_pred': ridge_pred, 'xgb_pred': xgb_pred}

    for model_name, pred in [('Ridge', ridge_pred), ('XGBoost', xgb_pred)]:
        idea2_rows.append({
            **eval_metrics(y_test, pred, target_label),
            'Model': model_name,
        })

idea2_df = pd.DataFrame(idea2_rows).set_index(['Target', 'Model'])
print('=== Idea 2: Regression Models ===')
display(
    idea2_df.style
    .format({'RMSE': '{:.4f}', 'Correlation': '{:.3f}', 'Dir. Accuracy': '{:.3f}'})
    .background_gradient(subset=['Dir. Accuracy'], cmap='RdYlGn')
    .background_gradient(subset=['Correlation'],   cmap='RdYlGn')
)


In [ ]:
# -- Scatter plots: Ridge and XGBoost predictions -------------------------
target_list = list(TARGET_LABELS.keys())
label_list  = list(TARGET_LABELS.values())
colors = ['steelblue', 'orange', 'grey']

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for col_i, (tcol, tlabel, color) in enumerate(zip(target_list, label_list, colors)):
    m = models_dict[tcol]
    y_true = m['y_test']

    for row_i, (model_name, pred) in enumerate(
        [('Ridge', m['ridge_pred']), ('XGBoost', m['xgb_pred'])]
    ):
        ax = axes[row_i][col_i]
        ax.scatter(pred, y_true, alpha=0.3, s=12, color=color)
        lo, hi = min(pred.min(), y_true.min()), max(pred.max(), y_true.max())
        ax.plot([lo, hi], [lo, hi], 'k--', linewidth=1, alpha=0.5)
        r  = np.corrcoef(pred, y_true)[0, 1]
        da = (np.sign(pred) == np.sign(y_true)).mean()
        ax.set_title(f'{model_name} | {tlabel}\nr={r:.3f}  Dir.Acc={da:.3f}', fontsize=9)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
        ax.axhline(0, color='grey', linewidth=0.4, linestyle='--')
        ax.axvline(0, color='grey', linewidth=0.4, linestyle='--')

plt.suptitle('Idea 2: Regression Predictions vs Actual (test period)', fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUTPUT_FIGS}/06_04_idea2_scatter.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# -- Feature importance: Ridge coefficients (始値 and 終値) ---------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, tcol, tlabel in zip(
    axes,
    ['open_ret', 'close_ret'],
    ['始値リターン (open_ret)', '終値リターン (close_ret)'],
):
    coef = pd.Series(
        models_dict[tcol]['ridge'].coef_,
        index=FEATURE_COLS
    ).sort_values()
    colors_bar = ['tomato' if v < 0 else 'steelblue' for v in coef.values]
    coef.plot(kind='barh', ax=ax, color=colors_bar)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'Ridge Coefficients — {tlabel}')
    ax.set_xlabel('Coefficient (standardised features)')

plt.tight_layout()
plt.savefig(f'{OUTPUT_FIGS}/06_05_ridge_coef.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# -- XGBoost feature importance (始値) ------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, tcol, tlabel in zip(
    axes,
    ['open_ret', 'close_ret'],
    ['始値リターン', '終値リターン'],
):
    imp = pd.Series(
        models_dict[tcol]['xgb'].feature_importances_,
        index=FEATURE_COLS
    ).sort_values(ascending=False)
    imp.plot(kind='barh', ax=ax, color='steelblue')
    ax.invert_yaxis()
    ax.set_title(f'XGBoost Feature Importance — {tlabel}')
    ax.set_xlabel('Importance score')

plt.tight_layout()
plt.savefig(f'{OUTPUT_FIGS}/06_06_xgb_importance.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. 両手法の比較と考察

### 予想されるパターン

| ターゲット | Idea 1 (ルールベース) | Idea 2 (回帰) | 理由 |
|------------|----------------------|---------------|------|
| 始値 | 比較的高い精度 | 最も高い精度 | 一夜の情報が始値に直接反映 |
| 終値 | やや低下 | やや低下 | 場中の日本固有要因が加わる |
| 場中 | 低い | 低い | 始値到達後の動きはADRで説明困難 |


In [ ]:
# -- Summary table: Idea 1 vs Idea 2 (best model per target) --------------
summary_rows = []

for tcol, tlabel in TARGET_LABELS.items():
    # Idea 1
    y_true = feat[tcol].values
    m1 = eval_metrics(y_true, synthetic_ret.values, tlabel)
    m1['Idea'] = 'Idea 1 (Rule-based)'
    summary_rows.append(m1)

    # Idea 2: XGBoost (usually better)
    m2 = eval_metrics(
        models_dict[tcol]['y_test'],
        models_dict[tcol]['xgb_pred'],
        tlabel
    )
    m2['Idea'] = 'Idea 2 (XGBoost)'
    summary_rows.append(m2)

summary = pd.DataFrame(summary_rows).set_index(['Idea', 'Target'])
print('=== Summary: Idea 1 vs Idea 2 ===')
display(
    summary.style
    .format({'RMSE': '{:.4f}', 'Correlation': '{:.3f}', 'Dir. Accuracy': '{:.3f}'})
    .background_gradient(subset=['Dir. Accuracy'], cmap='RdYlGn')
    .background_gradient(subset=['Correlation'],   cmap='RdYlGn')
)


In [ ]:
# -- Bar chart: Directional Accuracy by target and method -----------------
pivot = summary['Dir. Accuracy'].unstack('Target')

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(TARGET_COLS))
w = 0.35
bar1 = ax.bar(x - w/2, pivot.loc['Idea 1 (Rule-based)', TARGET_COLS],
              w, label='Idea 1 (Rule-based)', color='steelblue', alpha=0.85)
bar2 = ax.bar(x + w/2, pivot.loc['Idea 2 (XGBoost)',   TARGET_COLS],
              w, label='Idea 2 (XGBoost)',   color='orange',    alpha=0.85)

ax.axhline(0.5, color='red', linestyle='--', linewidth=1, label='Random (0.50)')
ax.set_xticks(x)
ax.set_xticklabels(['始値\nopen_ret', '終値\nclose_ret', '場中\nintraday_ret'], fontsize=11)
ax.set_ylabel('Directional Accuracy')
ax.set_ylim(0.3, 0.85)
ax.set_title('Directional Accuracy by Target and Method')
ax.legend()

for bar in [*bar1, *bar2]:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(f'{OUTPUT_FIGS}/06_07_dir_accuracy.png', dpi=150)
plt.show()


## 考察 / Discussion

### 結果の解釈

**始値リターン（open_ret）**
- 一夜の ADR 情報が最も直接的に反映される
- ルールベース・回帰モデルともに比較的高い相関・方向的中率が期待される
- これは「ギャップアップ/ダウン」の予測に相当する

**終値リターン（close_ret）**
- 始値リターンより精度が低下する傾向
- 開場後の日本固有ニュース・需給・機関投資家の動きが加わるため

**場中リターン（intraday_ret）**
- ADR データではほぼ予測不能（方向的中率 ≈ 0.50）
- 「一夜の情報は始値に効率的に織り込まれており、残りはノイズ」という  
  **効率的市場仮説と整合する結果**

### ルールベース vs 回帰

| 観点 | Idea 1 (ルールベース) | Idea 2 (回帰) |
|------|----------------------|---------------|
| 解釈性 | ◎ 日経の仕組みを反映 | ○ 係数・重要度で確認可 |
| 精度 | △ 上位10銘柄のみカバー | ○ データで最適化 |
| 前提知識 | 必要（価格ウェイト方式） | 少なくてよい |
| 拡張性 | △ 新銘柄追加が手動 | ◎ 特徴量追加で対応 |

### 次のステップ（Idea 3 候補）

- **日経先物（CME: NKD=F）の追加**: 最も直接的な予測変数。  
  先物を入れると他の変数の寄与が大幅に低下するため、  
  「ADR が先物に対して追加情報を持つか」という残差分析になる
- **Stacking アンサンブル**: Idea 1 の合成リターンを Idea 2 の特徴量として使用
- **銘柄数の拡大**: Nikkei 先物の構成銘柄を増やして上位 20〜30 銘柄をカバー
